# Fine-Tuning Pretrained ViT

Based on my trained CNN results and their dependence on the medical imaging protocol, I wanted to do something extra: is a pretrained model less dependent on quality of images?

The idea is that pretrained models, although not pretrained on medical images, have 'seen' so many images and have a better understanding of textures, colours, structures, resolution, etc. So by fine tuning a pretrained ViT on the original dataset, I wanted to see if it could still perform well on the transformed dataset. My assumption is that it will do better than the CNNs i trained as since the ViT has a better understanding of images, it should be less dependent on things like resolution, blur, etc. 

This wasn't part of my original study, but I was just curious to explore this idea and see whether it could be solution to overcome the imaging dependencies seen in the CNNs I trained by scratch.

(Note: I got most of my code online from preexisting fine tuning notebooks and walk throughs)

In [9]:
!pip install torch torchvision transformers datasets evaluate
!pip install "accelerate>=1.1.0"

from datasets import load_dataset
from transformers import ViTImageProcessor
import torch
from transformers import ViTForImageClassification, TrainingArguments, Trainer
from evaluate import load
import torch.nn as nn
from sklearn.metrics import roc_auc_score, classification_report
import numpy as np
from PIL import Image


### Transformations + Metrics

These functions convert the images into rgb (3 channels) so they align with the input format for the ViT; without doing this I was getting errors. And the other function computes metrics needed to see model performance.

In [10]:
def transform(example_batch):   #transform images for ViT
    images = [x.convert("RGB") for x in example_batch["image"]]  #have 3 channels for the ViT
    inputs = processor(images, return_tensors="pt")
    inputs["labels"] = example_batch["label"]
    return inputs

accuracy_metric = load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = logits.argmax(axis=1)
    probs = torch.softmax(torch.tensor(logits), dim=1)[:, 1].numpy()
    
    acc = accuracy_metric.compute(predictions=predictions, references=labels)
    auc = roc_auc_score(labels, probs)
    report = classification_report(labels, predictions, target_names=["NORMAL", "PNEUMONIA"], output_dict=True)
    
    return {"accuracy": acc["accuracy"], "auc": auc, "normal_recall": report["NORMAL"]["recall"], "pneumonia_recall": report["PNEUMONIA"]["recall"],}

Loading the dataset. I also chose to use this particular pretrained ViT because it was small enough for me to train on my laptop locally. I could have used googles pretrained ViT and trained on colab, however given how simple my CNNs were, I wanted to look at a simpler ViT.

In [ ]:
dataset = load_dataset("imagefolder", data_dir="../../chest_xray/xray_og") #getting original dataset
labels = dataset["train"].features["label"].names

model_name = "WinKawaks/vit-small-patch16-224"  #i am using this model since it was small enough for me to train on my laptop locally
processor = ViTImageProcessor.from_pretrained(model_name)

prepared_dataset = dataset.with_transform(transform)

In [14]:
train_labels = np.array(dataset["train"]["label"])
n_normal = (train_labels == 0).sum()
n_pneumonia  = (train_labels == 1).sum()
total = len(train_labels)

class_weights = torch.tensor([
    total / (2 * n_normal),       
    total / (2 * n_pneumonia),    
], dtype=torch.float)

In [15]:
#load model
model = ViTForImageClassification.from_pretrained(
    model_name,
    num_labels=len(labels),
    id2label={i: c for i, c in enumerate(labels)},
    label2id={c: i for i, c in enumerate(labels)},
    ignore_mismatched_sizes=True
)

class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels  = inputs.get("labels")
        outputs = model(**inputs)
        logits  = outputs.get("logits")
        loss    = nn.CrossEntropyLoss(weight=class_weights.to(logits.device))(logits, labels)
        return (loss, outputs) if return_outputs else loss

# setting to train on apple gpu
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
model.to(device)

[transformers] You passed `num_labels=2` which is incompatible to the `id2label` map of length `1000`.
Loading weights: 100%|██████████| 200/200 [00:00<00:00, 43907.92it/s]
[transformers] ViTForImageClassification LOAD REPORT from: WinKawaks/vit-small-patch16-224
Key               | Status   |                                                                                          
------------------+----------+------------------------------------------------------------------------------------------
classifier.weight | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([1000, 384]) vs model:torch.Size([2, 384])
classifier.bias   | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([1000]) vs model:torch.Size([2])          

Notes:
- MISMATCH:	ckpt weights were loaded, but they did not match the original empty weight shapes.


ViTForImageClassification(
  (vit): ViTModel(
    (embeddings): ViTEmbeddings(
      (patch_embeddings): ViTPatchEmbeddings(
        (projection): Conv2d(3, 384, kernel_size=(16, 16), stride=(16, 16))
      )
      (dropout): Dropout(p=0.0, inplace=False)
    )
    (layers): ModuleList(
      (0-11): 12 x ViTLayer(
        (attention): ViTAttention(
          (q_proj): Linear(in_features=384, out_features=384, bias=True)
          (k_proj): Linear(in_features=384, out_features=384, bias=True)
          (v_proj): Linear(in_features=384, out_features=384, bias=True)
          (o_proj): Linear(in_features=384, out_features=384, bias=True)
        )
        (layernorm_before): LayerNorm((384,), eps=1e-12, elementwise_affine=True, bias=True)
        (layernorm_after): LayerNorm((384,), eps=1e-12, elementwise_affine=True, bias=True)
        (mlp): ViTMLP(
          (activation_fn): GELUActivation()
          (fc1): Linear(in_features=384, out_features=1536, bias=True)
          (fc2): Linear

### Fine Tuning Model

I got the model arguments online. 

In [ ]:
#train args (got these online)
training_args = TrainingArguments(
    output_dir="../vit-finetuned",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-4,          
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    gradient_accumulation_steps=2,   
    num_train_epochs=5,              
    weight_decay=0.01,
    logging_dir="./logs",
    remove_unused_columns=False,
    dataloader_num_workers=0
)

trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=prepared_dataset["train"],
    eval_dataset=prepared_dataset["validation"],
    compute_metrics=compute_metrics,
)

#start fine tuning
trainer.train()


[transformers] `logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.
/Users/zubair/Desktop/ada_final_project/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss,Accuracy,Auc,Normal Recall,Pneumonia Recall
1,No log,2.956123,0.625000,1.000000,0.250000,1.000000
2,No log,0.569097,0.687500,1.000000,0.375000,1.000000
3,No log,0.002939,1.000000,1.000000,1.000000,1.000000
4,0.350564,0.081474,0.937500,1.000000,0.875000,1.000000
5,0.350564,0.490837,0.875000,1.000000,0.750000,1.000000


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  9.31it/s]
/Users/zubair/Desktop/ada_final_project/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  8.92it/s]
/Users/zubair/Desktop/ada_final_project/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  7.59it/s]
/Users/zubair/Desktop/ada_final_project/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
Writing model shards: 100%|███████

TrainOutput(global_step=815, training_loss=0.21961807970620373, metrics={'train_runtime': 720.3135, 'train_samples_per_second': 36.206, 'train_steps_per_second': 1.131, 'total_flos': 5.1034465082474496e+17, 'train_loss': 0.21961807970620373, 'epoch': 5.0})

### Evaluate Model

In [ ]:
metrics = trainer.evaluate()
print(metrics)

#save model
model.save_pretrained("../vit-finetuned-model")
processor.save_pretrained("../vit-finetuned-model")


/Users/zubair/Desktop/ada_final_project/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Training Loss,Validation Loss,Epoch,Accuracy,Auc,Normal Recall,Pneumonia Recall
0.350564,0.490837,5,0.875000,1.000000,0.750000,1.000000


{'eval_loss': 0.49083662033081055, 'eval_accuracy': 0.875, 'eval_auc': 1.0, 'eval_normal_recall': 0.75, 'eval_pneumonia_recall': 1.0}


Writing model shards: 100%|██████████| 1/1 [00:00<00:00, 11.13it/s]


['./vit-finetuned-model/preprocessor_config.json']